# 🥈 Silver — conform & clean (Lingua Foundry)

Combines the **app Lakehouse** (learner activity, `dbo` schema) with the
**Eventhouse** (enriched news, RTI hot path) into a clean, conformed Silver layer.

What it does:
- types timestamps & derives `*_date_key` join keys,
- standardises `language` / `cefr_level`,
- de-dupes news by `news_id` and **explodes the dynamic arrays**
  (`topic_tags`, `verbs`, `vocabulary`) into tidy bridge tables,
- drops the news `embedding` from the reporting copy.

Output → `LH_LanguageApp_Silver` (overwrite, idempotent). Schedule after ingestion.

In [ ]:
# ── Parameters (override in the Fabric pipeline / scheduler) ──────────────────
workspace            = "LanguageApp"          # Fabric workspace name OR GUID
bronze_lakehouse     = "LH_LanguageApp"       # app's operational lakehouse
silver_lakehouse     = "LH_LanguageApp_Silver"
gold_lakehouse       = "LH_LanguageApp_Gold"
bronze_schema        = "dbo"                   # app writes Delta under Tables/dbo

# Eventhouse (RTI) news source. Easiest: create a OneLake **shortcut** to the
# Eventhouse's `news_enriched` table inside the Silver lakehouse, then read it by
# name. Otherwise point news_onelake_path at the Eventhouse OneLake Delta path,
# or fall back to the Kusto connector below.
news_shortcut_table  = "news_enriched"         # table name in Silver lakehouse (via shortcut)
news_onelake_path    = ""                       # optional abfss:// path to the Delta table
eventhouse_query_uri = ""                        # optional: https://<cluster>.kusto.fabric.microsoft.com
eventhouse_database  = ""                        # optional KQL database name

In [ ]:
from pyspark.sql import functions as F, types as T

ONELAKE = "onelake.dfs.fabric.microsoft.com"

def tables_base(lakehouse: str) -> str:
    lh = lakehouse if (lakehouse.endswith(".Lakehouse") or "-" in lakehouse and len(lakehouse) == 36) else f"{lakehouse}.Lakehouse"
    return f"abfss://{workspace}@{ONELAKE}/{lh}/Tables"

def read_delta(lakehouse: str, table: str, schema: str | None = None):
    base = tables_base(lakehouse)
    path = f"{base}/{schema}/{table}" if schema else f"{base}/{table}"
    return spark.read.format("delta").load(path)

def write_delta(df, lakehouse: str, table: str, schema: str | None = None):
    base = tables_base(lakehouse)
    path = f"{base}/{schema}/{table}" if schema else f"{base}/{table}"
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true").save(path))
    print(f"  wrote {df.count():>7} -> {lakehouse}/{schema + '/' if schema else ''}{table}")

In [ ]:
def read_news():
    """Read enriched news from the Eventhouse, however it is surfaced."""
    # 1) OneLake shortcut table inside the Silver lakehouse (recommended).
    try:
        return read_delta(silver_lakehouse, news_shortcut_table)
    except Exception as e:
        print("shortcut read failed:", str(e)[:120])
    # 2) Explicit OneLake Delta path to the Eventhouse table.
    if news_onelake_path:
        return spark.read.format("delta").load(news_onelake_path)
    # 3) Kusto Spark connector (live query against the Eventhouse).
    if eventhouse_query_uri and eventhouse_database:
        return (spark.read.format("com.microsoft.kusto.spark.synapse.datasource")
                .option("kustoCluster", eventhouse_query_uri)
                .option("kustoDatabase", eventhouse_database)
                .option("kustoQuery", "news_enriched")
                .option("accessToken", mssparkutils.credentials.getToken(eventhouse_query_uri))
                .load())
    raise RuntimeError("No news source configured — set a shortcut, path, or Kusto URI.")

## Read bronze (app Lakehouse, `dbo`)

In [ ]:
users        = read_delta(bronze_lakehouse, "users",                 bronze_schema)
lessons      = read_delta(bronze_lakehouse, "lessons",               bronze_schema)
exercises    = read_delta(bronze_lakehouse, "exercises",             bronze_schema)
ex_scores    = read_delta(bronze_lakehouse, "exercise_scores",       bronze_schema)
convos       = read_delta(bronze_lakehouse, "conversations",         bronze_schema)
turns        = read_delta(bronze_lakehouse, "conversation_turns",    bronze_schema)
submissions  = read_delta(bronze_lakehouse, "worksheet_submissions", bronze_schema)
responses    = read_delta(bronze_lakehouse, "worksheet_responses",   bronze_schema)
date_dim     = read_delta(bronze_lakehouse, "date_dim",              bronze_schema)
print("bronze rows:", {t: df.count() for t, df in {
    "users": users, "lessons": lessons, "exercises": exercises,
    "exercise_scores": ex_scores, "conversations": convos, "turns": turns,
    "submissions": submissions, "responses": responses}.items()})

## Helpers: timestamp parsing & date keys

In [ ]:
def to_ts(col):
    return F.to_timestamp(col)

def date_key(ts_col):
    return F.date_format(ts_col, "yyyyMMdd").cast("int")

def norm_lang(col):
    return F.lower(F.trim(col))

def norm_cefr(col):
    return F.upper(F.trim(col))

## Conform learner tables

In [ ]:
silver_users = (users
    .withColumnRenamed("display_name", "user_name")
    .withColumn("native_language", norm_lang("native_language"))
    .withColumn("created_ts", to_ts("created_at"))
    .withColumn("created_date_key", date_key("created_ts"))
    .withColumn("is_sample", F.col("id").startswith("5a3b1e00"))
    .select("id", "user_name", "native_language", "created_ts", "created_date_key", "is_sample"))

silver_lessons = (lessons
    .withColumn("target_language", norm_lang("target_language"))
    .withColumn("difficulty", norm_cefr("difficulty"))
    .withColumn("created_ts", to_ts("created_at"))
    .withColumn("created_date_key", date_key("created_ts"))
    .select("id", "user_id", "target_language", "scenario", "mode", "verb",
            "grammar_focus", "difficulty", "created_ts", "created_date_key"))

silver_submissions = (submissions
    .withColumn("target_language", norm_lang("target_language"))
    .withColumn("difficulty", norm_cefr("difficulty"))
    .withColumn("submitted_ts", to_ts("submitted_at"))
    .withColumn("improvement", F.round(F.col("final_score_avg") - F.col("first_score_avg"), 4))
    .withColumn("accuracy_first", F.when(F.col("answered_count") > 0,
                F.round(F.col("first_correct_count") / F.col("answered_count"), 4)).otherwise(F.lit(0.0)))
    .withColumn("accuracy_final", F.when(F.col("answered_count") > 0,
                F.round(F.col("final_correct_count") / F.col("answered_count"), 4)).otherwise(F.lit(0.0))))

silver_responses = (responses
    .withColumn("target_language", norm_lang("target_language"))
    .withColumn("difficulty", norm_cefr("difficulty"))
    .withColumn("submitted_ts", to_ts("submitted_at")))

silver_exercise_scores = (ex_scores
    .withColumn("created_ts", to_ts("created_at"))
    .withColumn("date_key", date_key("created_ts")))

## Conform conversations (with `news_id`) + turn metrics

In [ ]:
turn_counts = (turns.groupBy("conversation_id")
    .agg(F.count("*").alias("turn_count"),
         F.sum(F.when(F.col("role") == "user", 1).otherwise(0)).alias("user_turns"),
         F.sum(F.when(F.col("corrected_text").isNotNull(), 1).otherwise(0)).alias("corrected_turns")))

silver_conversations = (convos
    .withColumn("target_language", norm_lang("target_language"))
    .withColumn("created_ts", to_ts("created_at"))
    .withColumn("ended_ts", to_ts("ended_at"))
    .withColumn("date_key", date_key("created_ts"))
    .withColumn("duration_min", F.round(
        (F.col("ended_ts").cast("long") - F.col("created_ts").cast("long")) / 60.0, 2))
    .withColumn("is_news_grounded", F.col("news_id").isNotNull())
    .join(turn_counts, convos.id == turn_counts.conversation_id, "left")
    .drop("conversation_id")
    .fillna({"turn_count": 0, "user_turns": 0, "corrected_turns": 0}))

## Conform news + explode dynamic arrays into bridges

In [ ]:
news_raw = read_news()

# Scalar news (drop the embedding from the reporting copy), de-dupe by news_id.
from pyspark.sql.window import Window
w = Window.partitionBy("news_id").orderBy(F.col("ingested_at").desc())
silver_news = (news_raw
    .withColumn("language", norm_lang("language"))
    .withColumn("cefr_level", norm_cefr("cefr_level"))
    .withColumn("seen_ts", to_ts("seen_at"))
    .withColumn("ingested_ts", to_ts("ingested_at"))
    .withColumn("seen_date_key", date_key("seen_ts"))
    .withColumn("_rn", F.row_number().over(w)).filter(F.col("_rn") == 1).drop("_rn")
    .select("news_id", "language", "cefr_level", "domain", "source_country",
            "title_translated", "title_original", "summary", "english_gloss",
            "seen_ts", "ingested_ts", "seen_date_key"))

def explode_bridge(col_name, out_col):
    return (news_raw.select("news_id", F.explode_outer(col_name).alias(out_col))
            .filter(F.col(out_col).isNotNull()))

silver_news_topic = explode_bridge("topic_tags", "topic").withColumn("topic", F.lower(F.trim("topic")))
silver_news_verb  = explode_bridge("verbs", "verb").withColumn("verb", F.lower(F.trim("verb")))
vocab = news_raw.select("news_id", F.explode_outer("vocabulary").alias("v")).filter(F.col("v").isNotNull())
silver_news_vocab = vocab.select("news_id",
        F.col("v.word").alias("word"), F.col("v.translation").alias("translation"))

## Write Silver

In [ ]:
print("Writing Silver tables…")
write_delta(silver_users,            silver_lakehouse, "users")
write_delta(silver_lessons,          silver_lakehouse, "lessons")
write_delta(silver_submissions,      silver_lakehouse, "worksheet_submissions")
write_delta(silver_responses,        silver_lakehouse, "worksheet_responses")
write_delta(silver_exercise_scores,  silver_lakehouse, "exercise_scores")
write_delta(silver_conversations,    silver_lakehouse, "conversations")
write_delta(date_dim,                silver_lakehouse, "date_dim")
write_delta(silver_news,             silver_lakehouse, "news")
write_delta(silver_news_topic,       silver_lakehouse, "news_topic")
write_delta(silver_news_verb,        silver_lakehouse, "news_verb")
write_delta(silver_news_vocab,       silver_lakehouse, "news_vocab")
print("Silver layer complete.")